<a href="https://colab.research.google.com/github/2000Voli/2000Voli/blob/main/Projeto_integrado_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Integrado — Feature Engineering
## QuantumFinance | MBA Data Science & IA — FIAP

**Objetivo:** aplicar técnicas de Feature Engineering à base de clientes da QuantumFinance, buscando melhorar a identificação de clientes com risco de inadimplência.

O dataset contém 50.000 registros de solicitações de cartão de crédito aprovadas. A variável-alvo identifica clientes que apresentaram atraso superior a 60 dias durante o primeiro ano após a contratação.

O projeto contempla análise exploratória, tratamento de dados, transformação de características, treinamento de classificadores e avaliação dos resultados.


In [ ]:

import pandas as pd

# Base oficial do Projeto Integrado
url = "https://raw.githubusercontent.com/diogenesjusto/FIAP/refs/heads/master/dados/credit_features_formatado.csv"

df = pd.read_csv(url)

# Verificar dimensões e primeiras linhas
print("Dimensões:", df.shape)
display(df.head())


In [ ]:
# Remover coluna de índice desnecessária
df = df.drop(columns=["Unnamed: 0"])

# Simplificar o nome da variável-alvo
df = df.rename(columns={
    "ROTULO_ALVO_MAU=1": "ROTULO_ALVO_MAU"
})

# Conferir o resultado
print("Dimensões:", df.shape)
print("\nDistribuição da variável-alvo:")
print(df["ROTULO_ALVO_MAU"].value_counts(normalize=True) * 100)

## 1. Análise exploratória dos dados

Nesta etapa, investigamos a estrutura do dataset, os tipos de variáveis, os valores ausentes e as principais características estatísticas.

A base original possui 50.000 registros e 55 colunas. Após a remoção da coluna de índice desnecessária, permanecem 54 colunas.

A variável-alvo apresenta a seguinte distribuição:

- **73,92%:** bons pagadores.
- **26,08%:** maus pagadores.

Também foram identificadas nove variáveis constantes, removidas por não apresentarem variação entre os clientes.

A análise estatística identificou valores extremos de renda e patrimônio, justificando a investigação de transformações dessas características.

In [ ]:
# Percentual de valores ausentes por variável
nulos = df.isnull().mean() * 100

# Mostrar apenas variáveis com valores ausentes
nulos = nulos[nulos > 0].sort_values(ascending=False)

print("Variáveis com valores ausentes:")
print(nulos.round(2))

In [ ]:
# Verificar os tipos de dados
print("Tipos de dados:")
print(df.dtypes.value_counts())

# Identificar variáveis numéricas e não numéricas
print("\nVariáveis numéricas:")
print(df.select_dtypes(include="number").columns.tolist())

print("\nVariáveis não numéricas:")
print(df.select_dtypes(exclude="number").columns.tolist())

In [ ]:
# Contar valores distintos por variável
valores_distintos = df.nunique(dropna=False)

# Identificar variáveis constantes
constantes = valores_distintos[valores_distintos == 1]

print("Variáveis constantes:")
print(constantes)

In [ ]:
# Remover as nove variáveis constantes
df = df.drop(columns=constantes.index)

# Conferir as dimensões
print("Dimensões após remoção:", df.shape)

In [ ]:
# Estatísticas descritivas das variáveis numéricas
display(df.describe().T.round(2))

In [ ]:
import matplotlib.pyplot as plt

# Histograma da renda mensal
plt.figure(figsize=(8, 4))

plt.hist(df["RENDA_PESSOAL_MENSAL"], bins=50)

plt.title("Distribuição da Renda Pessoal Mensal")
plt.xlabel("Renda mensal (R$)")
plt.ylabel("Quantidade de clientes")

plt.show()

## 2. Feature Engineering — Transformação de características

A análise exploratória revelou forte assimetria nas variáveis de renda e patrimônio.

Para reduzir a amplitude dos valores extremos, aplicamos a transformação logarítmica por meio de `np.log1p()`, criando duas novas características:

- **LOG_RENDA:** transformação logarítmica da renda pessoal mensal.
- **LOG_PATRIMONIO:** transformação logarítmica do patrimônio pessoal.

A transformação da renda facilitou a visualização da distribuição dos valores. Entretanto, no caso do patrimônio, a concentração de registros com valor zero permaneceu elevada.

As variáveis originais foram preservadas para permitir a comparação posterior.

In [ ]:
import numpy as np

# Criar uma nova variável com a renda transformada
df["LOG_RENDA"] = np.log1p(df["RENDA_PESSOAL_MENSAL"])

# Visualizar a nova distribuição
plt.figure(figsize=(8, 4))

plt.hist(df["LOG_RENDA"], bins=50)

plt.title("Distribuição da Renda após Transformação Logarítmica")
plt.xlabel("Log da renda mensal")
plt.ylabel("Quantidade de clientes")

plt.show()

In [ ]:
# Criar variável de patrimônio transformada
df["LOG_PATRIMONIO"] = np.log1p(df["VALOR_PATRIMONIO_PESSOAL"])

# Visualizar a distribuição
plt.figure(figsize=(8, 4))

plt.hist(df["LOG_PATRIMONIO"], bins=50)

plt.title("Distribuição do Patrimônio após Transformação Logarítmica")
plt.xlabel("Log do patrimônio pessoal")
plt.ylabel("Quantidade de clientes")

plt.show()

In [ ]:
# Percentual de clientes com patrimônio igual a zero
percentual_zero = (df["VALOR_PATRIMONIO_PESSOAL"] == 0).mean() * 100

print(f"Clientes com patrimônio zero: {percentual_zero:.2f}%")

### 2.1. Criação de uma variável indicadora

Identificamos que **95,22% dos clientes possuem patrimônio declarado igual a zero**.

Diante dessa concentração, criamos a variável binária `POSSUI_PATRIMONIO`, que assume:

- **0:** patrimônio declarado igual a zero.
- **1:** patrimônio declarado positivo.

Na análise exploratória, a taxa de inadimplência foi de 26,18% entre clientes sem patrimônio e 24,22% entre aqueles com patrimônio positivo.

Essa diferença sugere uma possível associação com o risco de crédito, mas não demonstra causalidade nem garante melhoria preditiva.

In [ ]:
# Criar indicador de patrimônio positivo
df["POSSUI_PATRIMONIO"] = (
    df["VALOR_PATRIMONIO_PESSOAL"] > 0
).astype(int)

# Verificar a distribuição
print(df["POSSUI_PATRIMONIO"].value_counts(normalize=True) * 100)

In [ ]:
# Taxa de inadimplência por grupo
taxa_inadimplencia = (
    df.groupby("POSSUI_PATRIMONIO")["ROTULO_ALVO_MAU"]
    .mean() * 100
)

print(taxa_inadimplencia.round(2))

In [ ]:
# Mediana da renda por classificação de risco
renda_por_grupo = df.groupby("ROTULO_ALVO_MAU")[
    "RENDA_PESSOAL_MENSAL"
].median()

print(renda_por_grupo)

In [ ]:
# Estatísticas da renda por grupo de inadimplência
comparacao_renda = df.groupby("ROTULO_ALVO_MAU")[
    "RENDA_PESSOAL_MENSAL"
].describe()

# Selecionar as principais estatísticas
display(
    comparacao_renda[["mean", "25%", "50%", "75%"]].round(2)
)

In [ ]:
# Comparar as duas variáveis
iguais = (
    df["QUANT_CONTAS_BANCARIAS"]
    == df["QUANT_CONTAS_BANCARIAS_ESPECIAIS"]
).all()

print("As duas variáveis são idênticas?", iguais)

In [ ]:
# Remover a variável redundante
df = df.drop(columns=["QUANT_CONTAS_BANCARIAS_ESPECIAIS"])

# Conferir as dimensões
print("Dimensões:", df.shape)

## 3. Preparação dos dados para modelagem

Antes do treinamento, removemos características redundantes e separamos as variáveis explicativas da variável-alvo.

O dataset foi dividido em:

- **70% para treinamento:** 35.000 registros.
- **30% para teste:** 15.000 registros.

Utilizamos `random_state=42` para garantir a reprodutibilidade da divisão e `stratify=y` para preservar a proporção de bons e maus pagadores.

Os procedimentos de imputação e normalização foram ajustados utilizando apenas os dados de treinamento, evitando que informações do conjunto de teste influenciassem o pré-processamento.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar variáveis explicativas e variável-alvo
X = df.drop(columns=["ID_CLIENTE", "ROTULO_ALVO_MAU"])
y = df["ROTULO_ALVO_MAU"]

# Separar treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

### 3.1. Tratamento dos valores ausentes

Foram identificadas oito variáveis com valores ausentes, incluindo campos profissionais com aproximadamente 67,57% de ausência.

Adotamos as seguintes estratégias:

- **Variáveis numéricas:** preenchimento pela mediana calculada no conjunto de treinamento.
- **Variáveis categóricas:** preenchimento com a categoria `DESCONHECIDO`.

A mediana foi escolhida por ser menos sensível a valores extremos.

Após o tratamento, os conjuntos de treinamento e teste não apresentaram valores ausentes.

**Limitação:** algumas variáveis numéricas representam códigos categóricos. O preenchimento pela mediana foi uma simplificação adotada neste experimento.

In [ ]:
# Identificar colunas numéricas com valores ausentes
colunas_nulas = X_treino.select_dtypes(
    include="number"
).columns

colunas_nulas = [
    coluna for coluna in colunas_nulas
    if X_treino[coluna].isnull().any()
]

# Calcular medianas apenas no treino
medianas = X_treino[colunas_nulas].median()

# Preencher treino e teste com as mesmas medianas
X_treino[colunas_nulas] = X_treino[colunas_nulas].fillna(medianas)
X_teste[colunas_nulas] = X_teste[colunas_nulas].fillna(medianas)

print("Variáveis tratadas:")
print(colunas_nulas)

print("\nValores ausentes restantes no treino:")
print(X_treino.isnull().sum().sum())

In [ ]:
# Identificar colunas categóricas
colunas_categoricas = X_treino.select_dtypes(
    include="object"
).columns

# Preencher valores ausentes
X_treino[colunas_categoricas] = (
    X_treino[colunas_categoricas].fillna("DESCONHECIDO")
)

X_teste[colunas_categoricas] = (
    X_teste[colunas_categoricas].fillna("DESCONHECIDO")
)

# Conferir valores ausentes
print("Nulos no treino:", X_treino.isnull().sum().sum())
print("Nulos no teste:", X_teste.isnull().sum().sum())

### 3.2. Codificação e seleção de características

A análise de cardinalidade identificou variáveis categóricas com poucas categorias e outras com milhares de valores distintos.

Para as variáveis de baixa cardinalidade, utilizamos `pd.get_dummies()`, conforme a técnica apresentada nas aulas.

As oito variáveis de alta cardinalidade restantes foram excluídas da primeira versão do modelo, evitando a criação de milhares de colunas.

Também foram removidas variáveis redundantes, após verificar a igualdade dos respectivos valores.

Essa seleção representa uma simplificação metodológica. A exclusão das variáveis de alta cardinalidade não significa que elas sejam irrelevantes para a previsão de inadimplência.

In [ ]:
# Contar categorias distintas no conjunto de treino
cardinalidade = (
    X_treino.select_dtypes(include="object")
    .nunique()
    .sort_values(ascending=False)
)

display(cardinalidade)

In [ ]:
# Identificar variáveis com mais de 30 categorias
alta_cardinalidade = cardinalidade[cardinalidade > 30]

# Identificar variáveis com até 30 categorias
baixa_cardinalidade = cardinalidade[cardinalidade <= 30]

print("Alta cardinalidade:")
print(alta_cardinalidade)

print("\nQuantidade de variáveis:")
print("Alta:", len(alta_cardinalidade))
print("Baixa:", len(baixa_cardinalidade))

In [ ]:
# Comparar os CEPs residencial e profissional
iguais = (
    X_treino["CEP_RESIDENCIAL_3"]
    == X_treino["CEP_PROFISSIONAL_3"]
).all()

print("Os CEPs são idênticos?", iguais)

In [ ]:
# Confirmar igualdade no conjunto de teste
iguais_teste = (
    X_teste["CEP_RESIDENCIAL_3"]
    == X_teste["CEP_PROFISSIONAL_3"]
).all()

print("CEPs idênticos no teste?", iguais_teste)

# Remover a coluna duplicada, caso sejam iguais
if iguais_teste:
    X_treino = X_treino.drop(columns=["CEP_PROFISSIONAL_3"])
    X_teste = X_teste.drop(columns=["CEP_PROFISSIONAL_3"])

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

In [ ]:
# Criar variáveis dummy para SEXO
dummies_treino = pd.get_dummies(X_treino["SEXO"], dtype=int)
dummies_teste = pd.get_dummies(X_teste["SEXO"], dtype=int)

# Manter no teste as mesmas colunas do treino
dummies_teste = dummies_teste.reindex(
    columns=dummies_treino.columns,
    fill_value=0
)

print("Categorias encontradas:")
print(dummies_treino.columns.tolist())

display(dummies_treino.head())

In [ ]:
# Contar registros por categoria, incluindo espaços em branco
print(
    X_treino["SEXO"].value_counts(dropna=False)
)

In [ ]:
# Substituir espaços em branco por DESCONHECIDO
X_treino["SEXO"] = X_treino["SEXO"].replace(
    r"^\s*$", "DESCONHECIDO", regex=True
)

X_teste["SEXO"] = X_teste["SEXO"].replace(
    r"^\s*$", "DESCONHECIDO", regex=True
)

# Verificar as categorias após o tratamento
print(X_treino["SEXO"].value_counts())

In [ ]:
# Criar variáveis dummy
dummies_treino = pd.get_dummies(
    X_treino["SEXO"], prefix="SEXO", dtype=int
)

dummies_teste = pd.get_dummies(
    X_teste["SEXO"], prefix="SEXO", dtype=int
)

# Garantir as mesmas colunas no treino e no teste
dummies_teste = dummies_teste.reindex(
    columns=dummies_treino.columns,
    fill_value=0
)

# Substituir a coluna original pelas variáveis dummy
X_treino = pd.concat(
    [X_treino.drop(columns=["SEXO"]), dummies_treino],
    axis=1
)

X_teste = pd.concat(
    [X_teste.drop(columns=["SEXO"]), dummies_teste],
    axis=1
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print("\nVariáveis criadas:")
print(dummies_treino.columns.tolist())

In [ ]:
# Contar categorias das variáveis de texto restantes
categorias_restantes = (
    X_treino.select_dtypes(include="object")
    .nunique()
    .sort_values()
)

display(categorias_restantes)

In [ ]:
# Selecionar variáveis com até 30 categorias
colunas_dummy = categorias_restantes[
    categorias_restantes <= 30
].index.tolist()

# Criar variáveis dummy
X_treino = pd.get_dummies(
    X_treino,
    columns=colunas_dummy,
    dtype=int
)

X_teste = pd.get_dummies(
    X_teste,
    columns=colunas_dummy,
    dtype=int
)

# Alinhar as colunas do teste às do treino
X_teste = X_teste.reindex(
    columns=X_treino.columns,
    fill_value=0
)

print("Variáveis transformadas:", colunas_dummy)
print("\nTreino:", X_treino.shape)
print("Teste:", X_teste.shape)

### 3.3. Normalização Min-Max

Após a codificação e seleção das características, aplicamos a normalização Min-Max.

O procedimento foi ajustado exclusivamente sobre o conjunto de treinamento e posteriormente aplicado ao conjunto de teste.

Ao final do pré-processamento, obtivemos:

- **Treinamento:** 35.000 registros e 125 características.
- **Teste:** 15.000 registros e 125 características.

Essa base foi utilizada para o treinamento dos classificadores.

In [ ]:
# Identificar as variáveis de texto restantes
alta_cardinalidade = X_treino.select_dtypes(
    include="object"
).columns.tolist()

# Remover essas variáveis da primeira versão
X_treino = X_treino.drop(columns=alta_cardinalidade)
X_teste = X_teste.drop(columns=alta_cardinalidade)

print("Variáveis removidas:", alta_cardinalidade)
print("\nTreino:", X_treino.shape)
print("Teste:", X_teste.shape)

## 4. Treinamento e avaliação dos classificadores

Para avaliar a qualidade das características preparadas, treinamos três classificadores apresentados nas aulas:

1. Regressão Logística.
2. Árvore de Decisão.
3. Random Forest.

A avaliação considerou acurácia, precision, recall, F1-score e matriz de confusão.

Como aproximadamente 74% dos clientes são bons pagadores, a acurácia isolada não é suficiente para avaliar a capacidade de identificar inadimplentes.

A Árvore de Decisão apresentou o maior recall da classe inadimplente entre os três modelos testados, sendo selecionada para o experimento comparativo de Feature Engineering.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Criar o escalonador
scaler = MinMaxScaler(feature_range=(0, 1))

# Ajustar no treino e aplicar aos dois conjuntos
X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

print("Treino:", X_treino_scaled.shape)
print("Teste:", X_teste_scaled.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Criar o modelo
modelo = LogisticRegression(max_iter=1000)

# Treinar com os dados de treino
modelo.fit(X_treino_scaled, y_treino)

# Realizar previsões no conjunto de teste
y_prev = modelo.predict(X_teste_scaled)

# Avaliar a acurácia
acuracia = accuracy_score(y_teste, y_prev)

print(f"Acurácia: {acuracia:.2%}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Relatório de classificação
print("Relatório de classificação:")
print(classification_report(y_teste, y_prev))

# Matriz de confusão
print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, y_prev))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# Criar e treinar a Árvore de Decisão
arvore = DecisionTreeClassifier(random_state=42)

arvore.fit(X_treino_scaled, y_treino)

# Realizar previsões
y_prev_arvore = arvore.predict(X_teste_scaled)

# Avaliar o modelo
print(classification_report(y_teste, y_prev_arvore))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Criar e treinar o modelo
floresta = RandomForestClassifier(random_state=42)

floresta.fit(X_treino_scaled, y_treino)

# Realizar previsões
y_prev_floresta = floresta.predict(X_teste_scaled)

# Avaliar o desempenho
print(classification_report(y_teste, y_prev_floresta))

## 5. Avaliação do impacto do Feature Engineering

Para avaliar o efeito das novas características, comparamos duas versões da Árvore de Decisão:

- **Sem Feature Engineering:** excluindo `LOG_RENDA`, `LOG_PATRIMONIO` e `POSSUI_PATRIMONIO`.
- **Com Feature Engineering:** incluindo essas três características.

Mantivemos o mesmo algoritmo, os mesmos parâmetros e a mesma divisão entre treinamento e teste.

### Resultados

| Métrica | Sem FE | Com FE |
|---|---:|---:|
| Recall dos inadimplentes | 31,44% | 32,31% |
| F1-score dos inadimplentes | 0,3082 | 0,3144 |
| Inadimplentes identificados | 1.230 | 1.264 |
| Falsos positivos | 2.841 | 2.865 |

As características criadas permitiram identificar **34 inadimplentes adicionais**, acompanhados de **24 falsos positivos adicionais**.

### Conclusão

O experimento demonstrou uma melhoria modesta no recall e no F1-score após a aplicação de Feature Engineering.

A variável `LOG_RENDA` também apareceu entre as características relevantes da Árvore de Decisão.

Entretanto, os resultados foram obtidos em uma única divisão treino/teste e não permitem afirmar que a melhoria seja estatisticamente significativa.

Além disso, o dataset contém apenas clientes cujas solicitações foram aprovadas, limitando a generalização dos resultados para todos os solicitantes de crédito.

Concluímos que as técnicas aplicadas contribuíram para a preparação do dataset e apresentaram um pequeno ganho preditivo na amostra avaliada.

In [ ]:
# Importância das características na Árvore de Decisão
importancias = pd.Series(
    arvore.feature_importances_,
    index=X_treino.columns
)

# Mostrar as 10 características mais importantes
display(importancias.sort_values(ascending=False).head(10))

In [ ]:
# Características criadas no projeto
novas_features = [
    "LOG_RENDA",
    "LOG_PATRIMONIO",
    "POSSUI_PATRIMONIO"
]

# Criar uma versão sem essas características
X_treino_sem_fe = X_treino.drop(columns=novas_features)
X_teste_sem_fe = X_teste.drop(columns=novas_features)

# Aplicar Min-Max sem utilizar informações do teste
scaler_sem_fe = MinMaxScaler()

X_treino_sem_fe = scaler_sem_fe.fit_transform(X_treino_sem_fe)
X_teste_sem_fe = scaler_sem_fe.transform(X_teste_sem_fe)

# Treinar Árvore de Decisão
arvore_sem_fe = DecisionTreeClassifier(random_state=42)
arvore_sem_fe.fit(X_treino_sem_fe, y_treino)

# Avaliar
y_prev_sem_fe = arvore_sem_fe.predict(X_teste_sem_fe)

print(classification_report(y_teste, y_prev_sem_fe))

In [ ]:
from sklearn.metrics import confusion_matrix

print("SEM Feature Engineering:")
print(confusion_matrix(y_teste, y_prev_sem_fe))

print("\nCOM Feature Engineering:")
print(confusion_matrix(y_teste, y_prev_arvore))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score

# Calcular recall da classe inadimplente
recall_sem = recall_score(y_teste, y_prev_sem_fe)
recall_com = recall_score(y_teste, y_prev_arvore)

# Comparar os resultados
plt.figure(figsize=(7, 4))

plt.bar(
    ["Sem Feature Engineering", "Com Feature Engineering"],
    [recall_sem * 100, recall_com * 100]
)

plt.ylabel("Recall dos inadimplentes (%)")
plt.title("Impacto do Feature Engineering na Árvore de Decisão")
plt.ylim(0, 40)

plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_sem = f1_score(y_teste, y_prev_sem_fe)
f1_com = f1_score(y_teste, y_prev_arvore)

print(f"F1-score sem Feature Engineering: {f1_sem:.4f}")
print(f"F1-score com Feature Engineering: {f1_com:.4f}")
print(f"Diferença: {f1_com - f1_sem:+.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

resultados = pd.DataFrame({
    "Modelo": ["Sem Feature Engineering", "Com Feature Engineering"],
    "Acurácia": [
        accuracy_score(y_teste, y_prev_sem_fe),
        accuracy_score(y_teste, y_prev_arvore)
    ],
    "Precision": [
        precision_score(y_teste, y_prev_sem_fe),
        precision_score(y_teste, y_prev_arvore)
    ],
    "Recall": [
        recall_score(y_teste, y_prev_sem_fe),
        recall_score(y_teste, y_prev_arvore)
    ],
    "F1-score": [
        f1_score(y_teste, y_prev_sem_fe),
        f1_score(y_teste, y_prev_arvore)
    ]
})

display(resultados.round(4))